In [ ]:
!apt-get update -qq
!apt-get install -y fonts-noto-core

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib import font_manager

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
words = open('devnagri_names.txt','r',encoding='utf-8').read().splitlines()
words[:8]

['आअबन्', 'आअभरन्', 'आअभस्', 'आअभत्', 'आअभीर्', 'आभीर्', 'आअभेर्', 'आअबि']

In [ ]:
len(words)

53982

In [ ]:
import re

def is_hindi(char):
    return char == '.' or '\u0900' <= char <= '\u097f'

# Re-build vocab with only the 'pure' bits
chars = sorted([c for c in set(''.join(words)) if is_hindi(c)])
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'ं', 2: 'ः', 3: 'अ', 4: 'आ', 5: 'इ', 6: 'ई', 7: 'उ', 8: 'ऊ', 9: 'ए', 10: 'ऐ', 11: 'ओ', 12: 'औ', 13: 'क', 14: 'ख', 15: 'ग', 16: 'घ', 17: 'च', 18: 'छ', 19: 'ज', 20: 'झ', 21: 'ञ', 22: 'ट', 23: 'ठ', 24: 'ड', 25: 'ढ', 26: 'ण', 27: 'त', 28: 'थ', 29: 'द', 30: 'ध', 31: 'न', 32: 'प', 33: 'फ', 34: 'ब', 35: 'भ', 36: 'म', 37: 'य', 38: 'र', 39: 'ऱ', 40: 'ल', 41: 'ळ', 42: 'ऴ', 43: 'व', 44: 'श', 45: 'ष', 46: 'स', 47: 'ह', 48: 'ा', 49: 'ि', 50: 'ी', 51: 'ु', 52: 'ू', 53: 'े', 54: 'ै', 55: 'ो', 56: 'ौ', 57: '्', 58: 'क़', 59: 'ख़', 60: 'ग़', 61: 'ज़', 62: 'फ़', 63: 'य़', 0: '.'}
64


In [ ]:
words_clean = []
for w in words:
    # Check if every character in the word (plus our '.' terminator) is in our mapping
    if all(ch in stoi for ch in w + '.'):
        words_clean.append(w)
    else:
        pass

print(f"Filtering complete. Kept {len(words_clean)} words out of {len(words)}.")

Filtering complete. Kept 43626 words out of 53982.


In [ ]:
# build the dataset
words = words_clean
block_size = 3

def build_dataset(words):
    X,Y = [],[]

    for w in words:
        context  =[0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix] # crop and append

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X,Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])        #80%
Xdev, Ydev = build_dataset(words[n1:n2])    #10%
Xte, Yte = build_dataset(words[n2:])        #10%

Xtr = Xtr.to(device)
Ytr = Ytr.to(device)

Xdev = Xdev.to(device)
Ydev = Ydev.to(device)

Xte = Xte.to(device)
Yte = Yte.to(device)

torch.Size([275341, 3]) torch.Size([275341])
torch.Size([34564, 3]) torch.Size([34564])
torch.Size([34207, 3]) torch.Size([34207])


In [ ]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator(device=device).manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocab_size, n_embd),            generator=g,device=device)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g,device=device) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g,device=device) * 0.1
# Layer 2
W2 = torch.randn((n_hidden, vocab_size),          generator=g,device=device) * 0.1
b2 = torch.randn(vocab_size,                      generator=g,device=device) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden),device=device)*0.1 + 1.0
bnbias = torch.randn((1, n_hidden),device=device)*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

max_steps = 200000
batch_size = 32
n = batch_size
lossi = []

for i in range(max_steps):

  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g,device=device)
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

  # forward pass
  emb = C[Xb] # embed the characters into vectors
  embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
  # Linear layer
  hprebn = embcat @ W1 + b1 # hidden layer pre-activation
  # BatchNorm layer
  bnmean = hprebn.mean(0, keepdim=True)
  bnvar = hprebn.var(0, keepdim=True, unbiased=True)
  bnvar_inv = (bnvar + 1e-5)**-0.5
  bnraw = (hprebn - bnmean) * bnvar_inv
  hpreact = bngain * bnraw + bnbias
  # Non-linearity
  h = torch.tanh(hpreact) # hidden layer
  logits = h @ W2 + b2 # output layer
  loss = F.cross_entropy(logits, Yb) # loss function

  # backward pass
  # for p in parameters:
  #   p.grad = None
  # loss.backward() # use this for correctness comparisons, delete it later!

  # manual backprop
  # 1. Backprop through Cross Entropy (Compacted version)
  dlogits = F.softmax(logits, 1)
  dlogits[range(n), Yb] -= 1
  dlogits /= n

  # 2. Layer 2
  dh = dlogits @ W2.T
  dW2 = h.T @ dlogits
  db2 = dlogits.sum(0)

  # 3. Non-linearity
  dhpreact = (1.0 - h**2) * dh

  # 4. BatchNorm manual backprop
  dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
  dbnbias = dhpreact.sum(0, keepdim=True)
  dbnraw = bngain * dhpreact

  current_bndiff = hprebn - bnmean

  dbnvar_inv = (current_bndiff * dbnraw).sum(0, keepdim=True)
  dbnvar = (-0.5 * (bnvar + 1e-5)**-1.5) * dbnvar_inv

  dbndiff = bnvar_inv * dbnraw # branch 1
  dbndiff += (2.0 / (n-1)) * current_bndiff * dbnvar # branch 2 (accumulated)

  # 5. BatchNorm Mean branch
  dbnmean = -dbndiff.sum(0, keepdim=True)
  dhprebn = dbndiff.clone() + (1.0/n) * dbnmean

  # 6. Layer 1 & Embedding
  dembcat = dhprebn @ W1.T
  dW1 = embcat.T @ dhprebn
  db1 = dhprebn.sum(0)
  demb = dembcat.view(emb.shape)
  dC = torch.zeros_like(C,device=device)
  for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
      ix = Xb[k,j]
      dC[ix] += demb[k,j]

  grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]

  # update
  lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
  for p, grad in zip(parameters, grads):
    p.data += -lr * grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())

20104
      0/ 200000: 4.2877
  10000/ 200000: 2.6184
  20000/ 200000: 2.3140
  30000/ 200000: 2.4090
  40000/ 200000: 2.2722
  50000/ 200000: 1.9644
  60000/ 200000: 2.2091
  70000/ 200000: 2.0795
  80000/ 200000: 1.9542
  90000/ 200000: 2.2472
 100000/ 200000: 2.4498
 110000/ 200000: 2.1584
 120000/ 200000: 2.1771
 130000/ 200000: 1.6631
 140000/ 200000: 1.7914
 150000/ 200000: 2.0486
 160000/ 200000: 1.7838
 170000/ 200000: 1.9239
 180000/ 200000: 1.7570
 190000/ 200000: 2.3557


In [ ]:
# calibrate the batch norm at the end of training

with torch.no_grad():
  # pass the training set through
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 + b1
  # measure the mean/std over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnvar = hpreact.var(0, keepdim=True, unbiased=True)


In [ ]:
# evaluate train and val loss

@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 + b1
  hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.0504837036132812
val 2.0489163398742676


In [ ]:
# sample from the model
g = torch.Generator(device=device).manual_seed(2147483647 + 10)

for _ in range(20):

    out = []
    context = [0] * block_size
    while True:
      # forward pass
      emb = C[torch.tensor([context])] # (1,block_size,d)
      embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
      h = torch.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # sample
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break

    print(''.join(itos[i] for i in out))

ऱाशिरजन्.
णैव्श्.
छशन्कल्न.
छिदुशिनि.
आएनि.
डिनन्.
ऊषन्गोनि.
ज़स्रिथन्.
ःएअयह्.
ऱमत्.
ख़निनि.
आअथिज्वित्.
ऱश्मिथ.
आवश्वि.
ऊघ्लन.
ख़ार्नथि.
ख़लेरन्च्य.
ख़िलिनि.
ऱविथ्य.
णागेन्थ्.


In [ ]:
from sympy import ln
-ln(1/vocab_size)

In [ ]:

font_path = "/usr/share/fonts/truetype/noto/NotoSansDevanagari-Regular.ttf"
prop = font_manager.FontProperties(fname=font_path)

C_cpu = C.detach().cpu()

plt.figure(figsize=(10,10))
plt.scatter(C_cpu[:,0], C_cpu[:,1], s=400, c='royalblue', alpha=0.6)

for i in range(C_cpu.shape[0]):
    plt.text(C_cpu[i,0].item(), C_cpu[i,1].item(), itos[i],
             ha="center", va="center",
             fontsize=12,
             fontproperties=prop)

plt.title("The Model's Mental Map of Hindi", fontproperties=prop, fontsize=15)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()
